# Colab 動作確認（合成データ・約10分）

本番の `train_on_colab.ipynb` と同じ手順（インストール → 学習 → デバッグ）を、ノートブックに埋め込んだ小さな合成データセット（6エピソード）で確かめます。
**ランタイム → ランタイムのタイプを変更 → T4 GPU** にしてから **ランタイム → すべてのセルを実行** してください。最後に `SMOKE TEST: OK` が出れば成功です。Google Drive は使いません。

In [ ]:
# GPU が割り当てられているか確認（CPU のままだと学習が数十倍遅い）
import torch, sys, platform
print(sys.version.split()[0], platform.platform())
if not torch.cuda.is_available():
    raise SystemExit("GPU がありません。メニューの「ランタイム → ランタイムのタイプを変更」で T4 GPU を選び、"
                     "「ランタイム → すべてのセルを実行」をやり直してください。")
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# Colab の PyTorch はそのまま使い、LeRobot 本体と必要な依存だけ入れる
!pip install -q --no-deps "lerobot @ git+https://github.com/huggingface/lerobot.git@768e36660d1408c71118a2760f831c037fbfa17d"
!pip install -q "datasets==3.6.0" "huggingface_hub==0.32.3" "draccus==0.10.0" einops jsonlines termcolor deepdiff "av>=12" "imageio[ffmpeg]" safetensors pyyaml

# 動画の読み込みには TorchCodec が必要（torchvision 0.26 以降は VideoReader が削除されたため）
# 入っている PyTorch に対応する版を選ぶ（https://github.com/meta-pytorch/torchcodec の対応表）
import torch
mm = ".".join(torch.__version__.split("+")[0].split(".")[:2])
codec = {"2.7": "0.5", "2.8": "0.7", "2.9": "0.9", "2.10": "0.10", "2.11": "0.11"}.get(mm)
spec = f"torchcodec=={codec}.*" if codec else "torchcodec"
print("torch", torch.__version__, "→", spec)
!pip install -q "{spec}"

import lerobot, torchcodec
from torchcodec.decoders import VideoDecoder
print("torch", torch.__version__, "| torchcodec", torchcodec.__version__, "| cuda", torch.cuda.is_available())

In [ ]:
# 埋め込みデータ（コード一式 + 合成データセット, 120 KB）を展開
import base64, io, zipfile, os
PAYLOAD = '...'
os.makedirs('/content/smoke', exist_ok=True)
zipfile.ZipFile(io.BytesIO(base64.b64decode(PAYLOAD))).extractall('/content/smoke')
!ls /content/smoke /content/smoke/crane_vision_pick_synth/meta

In [ ]:
%%bash
cd /content/smoke/pai-crane-vision-pick/training
sed -e 's/val_every: 1000/val_every: 100/; s/log_every: 100/log_every: 25/' configs/vision_train.yaml > /content/smoke/smoke_train.yaml
python scripts/train_vision.py --config /content/smoke/smoke_train.yaml --steps 300 \
    --dataset-root /content/smoke/crane_vision_pick_synth --output /content/smoke/ckpt 2>&1 | grep -v -i warn

In [ ]:
%%bash
cd /content/smoke/pai-crane-vision-pick/training
python scripts/debug_policy_vision.py --dataset-root /content/smoke/crane_vision_pick_synth --repo-id crane_vision_pick \
    --policy-path /content/smoke/ckpt --episode-index 0 5 --device auto 2>&1 | grep -v -i warn

In [ ]:
import os, json, torch, torchcodec
ok = os.path.exists('/content/smoke/ckpt/model.safetensors') and os.path.exists('/content/smoke/ckpt/debug_ep0.png')
s = json.load(open('/content/smoke/ckpt/train_settings.json')) if ok else {}
print('torch', torch.__version__, '| torchcodec', torchcodec.__version__, '| device', s.get('device_used'),
      '| minutes', s.get('minutes'), '| best_val_l1', s.get('best_val_l1'))
print('SMOKE TEST:', 'OK' if ok else 'FAILED')
from IPython.display import Image, display
if ok: display(Image('/content/smoke/ckpt/debug_ep0.png', width=600))